<a href="https://colab.research.google.com/github/ThinkingBeyond/BeyondAI-2025/blob/main/Aryan%20Basnet%2C%20Arnav%20Maharjan%20and%20Ashila%20A%20M%20Ardiyansyah/04_dataset4_LMIC_TB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NOTE ON RUNTIME AND OUTPUTS**

# Google Colab may have disconnected or reset the runtime during long training sessions, crashes, or memory interruptions. When this occurred, some previously displayed outputs in the notebook were no longer visible. However, all results remained saved and logged correctly. Each model’s complete metrics and metadata were stored as JSON files in my Google Drive folder:

# [https://drive.google.com/drive/folders/1ejlJaZhHEBm-1khLBJ--mbG2pg5TZHoJ?usp=sharing](https://drive.google.com/drive/folders/1ejlJaZhHEBm-1khLBJ--mbG2pg5TZHoJ?usp=sharing)

# These JSON files contain the full and reliable outputs for all models across all datasets, even if certain notebook outputs were lost due to runtime resets.

# ==============================
# SETUP: Freeze all package versions
# ==============================
Ensure reproducibility by installing the exact versions of packages used in these notebooks. This includes pre-installed packages in Colab.

The packages and versions used are:

- numpy==1.25.2
- pandas==2.1.1
- matplotlib==3.8.0
- seaborn==0.12.2
- scikit-learn==1.3.2
- tensorflow==2.15.0
- keras==2.15.0
- scipy==1.11.2
- opencv-python==4.9.0.73
- Pillow==10.0.1
- h5py==3.9.0
- google-colab==2.0.0

In [ ]:
# ==========================================
# CHEST X-RAY CLASSIFICATION - TB DATASET
# DATASET: Dataset of Tuberculosis Chest X-rays Images
# COUNTRY INCOME LEVEL: LMIC
# ==========================================

# STEP 1: METADATA
DATASET_NAME = "dataset_tb_chest_xray"
COUNTRY_INCOME_LEVEL = "LMIC"  # Pakistan
NUM_CLASSES = 2  # Normal, TB

print(f"Dataset: {DATASET_NAME}")
print(f"Income Level: {COUNTRY_INCOME_LEVEL}")
print(f"Classes: Normal, TB")

# STEP 2: MOUNT GOOGLE DRIVE
# Required to save results and access uploaded files in Colab
from google.colab import drive
drive.mount('/content/drive')

# Create results directory in Drive if it doesn't exist
!mkdir -p /content/drive/MyDrive/xray_research_results

# STEP 3: IMPORT REQUIRED LIBRARIES
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import json
import time
import gc
import warnings
warnings.filterwarnings('ignore')  # suppress warnings for clean output

# STEP 4: UPLOAD & EXTRACT LOCAL DATASET
# Assumes 'tb_chest_xray.zip' is uploaded in Colab
zip_path = "/content/Dataset of Tuberculosis Chest X-rays Images.zip"
extract_path = "/content/dataset_tb_chest_xray"

# Extract zip contents
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dataset extracted to: {extract_path}")

# Preview dataset structure to understand folder hierarchy
print("\nDataset structure preview:")
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    if level < 3:  # limit depth preview for readability
        for d in dirs[:5]:
            print(f"{subindent}{d}/")
        if len(files) > 0:
            print(f"{subindent}... {len(files)} files")

# STEP 5: SETUP DATA PATHS AND SPLIT DATA
from sklearn.model_selection import train_test_split
import shutil

# Original dataset directories
top_folder = os.path.join(extract_path, "Dataset of Tuberculosis Chest X-rays Images")
normal_dir = os.path.join(top_folder, "Normal Chest X-rays")
tb_dir = os.path.join(top_folder, "TB Chest X-rays")

# Output directories for split data
base_dir = "/content/dataset_tb_split"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")
test_dir = os.path.join(base_dir, "test")

# Create subfolders for each split and class
for d in [train_dir, val_dir, test_dir]:
    for c in ['normal', 'tb']:
        os.makedirs(os.path.join(d, c), exist_ok=True)

# Function to split files and copy to respective directories
def split_copy(src_dir, class_name):
    files = os.listdir(src_dir)
    # Split 70% train, 15% val, 15% test
    train_files, temp_files = train_test_split(files, test_size=0.3, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

    for f in train_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(train_dir, class_name, f))
    for f in val_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(val_dir, class_name, f))
    for f in test_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(test_dir, class_name, f))

# Apply split for each class
split_copy(normal_dir, 'normal')
split_copy(tb_dir, 'tb')

print("\nData split complete.")

# Check number of images per class in each split
print("\nImage count per class:")
for split in ['train', 'val', 'test']:
    print(f"\n{split.upper()}:")
    split_path = os.path.join(base_dir, split)
    for class_name in ['normal', 'tb']:
        count = len(os.listdir(os.path.join(split_path, class_name)))
        print(f"  {class_name}: {count} images")

# STEP 6: DATA PREPROCESSING
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
CLASS_NAMES = ['normal', 'tb']

# Training augmentation for better generalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1],
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# Validation and test generators only rescale pixel values
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators to feed data to models
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# STEP 7: COMPUTE CLASS WEIGHTS
# Handles class imbalance to prevent bias toward majority class
y_train = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {i: weight for i, weight in enumerate(class_weights_array)}

# Display training class distribution and computed weights
print(f"\nClass distribution in training:")
for class_name, class_idx in sorted(train_generator.class_indices.items(), key=lambda x: x[1]):
    count = np.sum(y_train == class_idx)
    print(f"  {class_name} ({class_idx}): {count} samples")
print(f"\nClass weights computed: {class_weights}")

# STEP 8: DEFINE MODEL ARCHITECTURES

# Simple baseline CNN
def create_baseline_cnn(input_shape=(224,224,3), num_classes=2):
    model = keras.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Transfer learning model (MobileNetV2, EfficientNetB0, ResNet50)
def create_transfer_model(base_model_name, input_shape=(224,224,3), num_classes=2):
    if base_model_name == 'MobileNetV2':
        base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'EfficientNetB0':
        base = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'ResNet50':
        base = tf.keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights='imagenet')
    else:
        raise ValueError(f"Unknown model: {base_model_name}")

    base.trainable = False  # freeze pretrained layers
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    return model

# STEP 9: TRAINING & EVALUATION FUNCTION
def train_and_evaluate(model, model_name, train_gen, val_gen, test_gen, class_weights):
    print(f"\n{'='*50}\nTraining {model_name}\n{'='*50}")

    # Compile model with categorical crossentropy for multi-class output
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Callbacks for early stopping and learning rate reduction
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=1,
        restore_best_weights=True,
        verbose=1
    )
    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )

    # Train model and record time
    start_time = time.time()
    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[early_stop, reduce_lr],
        class_weight=class_weights,
        verbose=1
    )
    training_time = (time.time() - start_time)/60  # in minutes

    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_loss, test_acc = model.evaluate(test_gen, verbose=0)

    # Predict class probabilities and convert to label indices
    predictions = model.predict(test_gen)
    y_pred = np.argmax(predictions, axis=1)
    y_true = test_gen.classes

    # Compute per-class and weighted F1 scores
    f1_per_class = f1_score(y_true, y_pred, average=None).tolist()
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)

    # Map indices to class names
    idx_to_name = {v:k for k,v in test_gen.class_indices.items()}
    class_names_ordered = [idx_to_name[i] for i in range(NUM_CLASSES)]

    # Prepare results dictionary
    results = {
        'dataset_name': DATASET_NAME,
        'country_income': COUNTRY_INCOME_LEVEL,
        'model_name': model_name,
        'num_classes': NUM_CLASSES,
        'class_names': class_names_ordered,
        'f1_per_class': f1_per_class,
        'f1_weighted': float(f1_weighted),
        'confusion_matrix': cm.tolist(),
        'training_time_minutes': float(training_time),
        'num_images_train': train_gen.samples,
        'num_images_val': val_gen.samples,
        'num_images_test': test_gen.samples,
        'num_parameters': int(model.count_params()),
        'test_accuracy': float(test_acc),
        'epochs_trained': len(history.history['loss'])
    }

    # Display results
    print(f"\n{model_name} Results:")
    print(f"F1 Scores per class: {f1_per_class}")
    for i, name in enumerate(class_names_ordered):
        print(f"  {name}: {f1_per_class[i]:.4f}")
    print(f"Weighted F1 Score: {f1_weighted:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Training time: {training_time:.2f} minutes")
    print(f"Parameters: {model.count_params():,}")

    # Save results to JSON in Drive
    filename = f'/content/drive/MyDrive/xray_research_results/{DATASET_NAME}_{COUNTRY_INCOME_LEVEL}_{model_name}_results.json'
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Results saved to: {filename}")

    # Clean up to free memory
    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return results

# STEP 10: TRAIN ALL MODELS
all_results = []

for i, model_name in enumerate(['BaselineCNN', 'MobileNetV2', 'EfficientNetB0', 'ResNet50'], 1):
    print(f"\n{'='*60}\nTRAINING MODEL {i}/4: {model_name}\n{'='*60}")
    if model_name == 'BaselineCNN':
        model = create_baseline_cnn(num_classes=NUM_CLASSES)
    else:
        model = create_transfer_model(model_name, num_classes=NUM_CLASSES)
    results = train_and_evaluate(model, model_name, train_generator, validation_generator, test_generator, class_weights)
    all_results.append(results)

# STEP 11: FINAL SUMMARY
print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)

for result in all_results:
    print(f"\n{result['model_name']}:")
    print(f"  Weighted F1: {result['f1_weighted']:.4f}")
    print(f"  F1 per class: {result['f1_per_class']}")
    print(f"  Training Time: {result['training_time_minutes']:.2f} min")
    print(f"  Parameters: {result['num_parameters']:,}")
    print(f"  Epochs trained: {result['epochs_trained']}")

print(f"\nAll results saved to: /content/drive/MyDrive/xray_research_results/")

Dataset: dataset_tb_chest_xray
Income Level: LMIC
Classes: Normal, TB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extracted to: /content/dataset_tb_chest_xray

Dataset structure preview:
dataset_tb_chest_xray/
  Dataset of Tuberculosis Chest X-rays Images/
  Dataset of Tuberculosis Chest X-rays Images/
    Normal Chest X-rays/
    TB Chest X-rays/
    Normal Chest X-rays/
      ... 514 files
    TB Chest X-rays/
      ... 2494 files

Data split complete.

Image count per class:

TRAIN:
  normal: 359 images
  tb: 1745 images

VAL:
  normal: 77 images
  tb: 374 images

TEST:
  normal: 78 images
  tb: 375 images
Found 2104 images belonging to 2 classes.
Found 451 images belonging to 2 classes.
Found 453 images belonging to 2 classes.

Class distribution in training:
  normal (0): 359 samples
  tb (1): 1745 samples

Class weights computed: {0: np.float64(2.9303621169916436), 1: np.float64(0.6028653

In [ ]:
# ==========================================
# CHEST X-RAY CLASSIFICATION - TB DATASET
# MODELS: EfficientNetB0 & ResNet50 (Optimized for Small / Imbalanced Data)
# ==========================================

# STEP 1: METADATA
# Basic dataset information and class definitions
DATASET_NAME = "dataset_tb_chest_xray"
COUNTRY_INCOME_LEVEL = "LMIC"
NUM_CLASSES = 2
CLASS_NAMES = ['normal', 'tb']

# STEP 2: MOUNT GOOGLE DRIVE
# Force remount to ensure access to Drive for saving results
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Create directory to store training results
import os
os.makedirs('/content/drive/MyDrive/xray_research_results', exist_ok=True)

# STEP 3: IMPORT REQUIRED LIBRARIES
# Core libraries for file handling, timing, memory management, deep learning, and metrics
import zipfile, shutil, time, gc, json, warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')  # suppress warnings for clarity

# STEP 4: EXTRACT DATASET
# Extract uploaded zip file containing chest X-ray images
zip_path = "/content/Dataset of Tuberculosis Chest X-rays Images.zip"
extract_path = "/content/dataset_tb_chest_xray"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Paths to class-specific folders
normal_dir = os.path.join(extract_path, "Dataset of Tuberculosis Chest X-rays Images", "Normal Chest X-rays")
tb_dir = os.path.join(extract_path, "Dataset of Tuberculosis Chest X-rays Images", "TB Chest X-rays")

# STEP 5: SPLIT DATASET
# Define output directories for train, val, test splits
base_dir = "/content/dataset_tb_split"
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")
test_dir = os.path.join(base_dir, "test")

# Create subdirectories for each class in all splits
for d in [train_dir, val_dir, test_dir]:
    for c in CLASS_NAMES:
        os.makedirs(os.path.join(d, c), exist_ok=True)

# Function to split files into train/val/test and copy them
def split_copy(src_dir, class_name):
    files = os.listdir(src_dir)
    # Split 70% train, 15% val, 15% test
    train_files, temp_files = train_test_split(files, test_size=0.3, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)
    # Copy files to respective folders
    for f in train_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(train_dir, class_name, f))
    for f in val_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(val_dir, class_name, f))
    for f in test_files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(test_dir, class_name, f))

# Apply split for both classes
split_copy(normal_dir, 'normal')
split_copy(tb_dir, 'tb')

# STEP 6: DATA AUGMENTATION
# Define image size, batch size, and epochs
IMG_SIZE = (224, 224)
BATCH_SIZE = 8  # small batch due to small dataset
EPOCHS = 20

# Training augmentation to increase robustness
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    brightness_range=[0.8,1.2],
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation and test generators only rescale pixel values
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators for Keras model training
train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)
validation_generator = val_test_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_generator = val_test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

# STEP 7: COMPUTE CLASS WEIGHTS
# Balances loss function to account for class imbalance
y_train = train_generator.classes
class_weights_array = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {i: weight for i, weight in enumerate(class_weights_array)}
print(f"\nClass weights: {class_weights}")

# STEP 8: MODEL CREATION (Smaller fine-tuning)
def create_transfer_model(base_model_name, input_shape=(224,224,3), num_classes=2):
    # Select pretrained base model
    if base_model_name == 'EfficientNetB0':
        base = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights='imagenet')
    elif base_model_name == 'ResNet50':
        base = tf.keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights='imagenet')
    else:
        raise ValueError(f"Unknown model: {base_model_name}")

    # Freeze 85% of layers to prevent overfitting on small dataset
    base.trainable = True
    freeze_until = int(len(base.layers) * 0.85)
    for layer in base.layers[:freeze_until]:
        layer.trainable = False

    # Add classification head
    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    return model

# STEP 9: TRAINING & EVALUATION
def train_and_evaluate(model, model_name):
    print(f"\n{'='*50}\nTraining {model_name}\n{'='*50}")

    # Compile with small learning rate for fine-tuning
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # Callbacks for early stopping and LR reduction
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)

    start_time = time.time()
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=validation_generator,
        class_weight=class_weights,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    training_time = (time.time() - start_time)/60  # minutes

    # Evaluate on test set
    predictions = model.predict(test_generator)
    y_pred = np.argmax(predictions, axis=1)
    y_true = test_generator.classes

    f1_per_class = f1_score(y_true, y_pred, average=None).tolist()
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)

    # Prepare results dictionary
    results = {
        'dataset_name': DATASET_NAME,
        'country_income': COUNTRY_INCOME_LEVEL,
        'model_name': model_name,
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'f1_per_class': f1_per_class,
        'f1_weighted': float(f1_weighted),
        'confusion_matrix': cm.tolist(),
        'training_time_minutes': float(training_time),
        'num_images_train': train_generator.samples,
        'num_images_val': validation_generator.samples,
        'num_images_test': test_generator.samples,
        'num_parameters': int(model.count_params()),
        'epochs_trained': len(history.history['loss'])
    }

    # Save results as JSON
    filename = f'/content/drive/MyDrive/xray_research_results/{DATASET_NAME}_{COUNTRY_INCOME_LEVEL}_{model_name}_results.json'
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)

    # Print concise summary
    print(f"\n{model_name} Results:")
    print(f"Weighted F1: {f1_weighted:.4f}, F1 per class: {f1_per_class}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Training time: {training_time:.2f} min, Params: {model.count_params():,}")

    # Clean up to release GPU memory
    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return results

# STEP 10: TRAIN BOTH MODELS
all_results = []
for model_name in ['EfficientNetB0', 'ResNet50']:
    model = create_transfer_model(model_name)
    results = train_and_evaluate(model, model_name)
    all_results.append(results)

# STEP 11: SUMMARY
print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)
for r in all_results:
    print(f"\n{r['model_name']}: Weighted F1: {r['f1_weighted']:.4f}, Epochs: {r['epochs_trained']}, Params: {r['num_parameters']:,}")
print(f"\nAll results saved to: /content/drive/MyDrive/xray_research_results/")

Mounted at /content/drive
Found 2104 images belonging to 2 classes.
Found 451 images belonging to 2 classes.
Found 453 images belonging to 2 classes.

Class weights: {0: np.float64(2.9303621169916436), 1: np.float64(0.602865329512894)}

Training EfficientNetB0
Epoch 1/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 73s 165ms/step - accuracy: 0.4627 - loss: 0.7510 - val_accuracy: 0.8293 - val_loss: 0.6751 - learning_rate: 1.0000e-04
Epoch 2/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 31s 119ms/step - accuracy: 0.4413 - loss: 0.7401 - val_accuracy: 0.8293 - val_loss: 0.6812 - learning_rate: 1.0000e-04
Epoch 3/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 33s 126ms/step - accuracy: 0.5471 - loss: 0.7147 - val_accuracy: 0.8293 - val_loss: 0.6668 - learning_rate: 1.0000e-04
Epoch 4/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 31s 118ms/step - accuracy: 0.5401 - loss: 0.7011 - val_accuracy: 0.1707 - val_loss: 0.7157 - learning_rate: 1.0000e-04
Epoch 5/20
263/263 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.4325 - loss: 0.7060
Epoch 5: Reduce